# Backpropagation through time
1. A one-unit RNN on three numbers: every BPTT term by hand, checked against TensorFlow.
2. The 3-unit RNN on the toy reviews: a NumPy BPTT checked against TensorFlow.
3. A real IMDB review: the gradient of the shared W_h equals the sum of one term per time step.
4. Gradient descent with these gradients trains the toy RNN.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. One hidden unit, three time steps, by hand
Scalars: w_i = 0.5, w_h = 0.8, w_o = 1.0, no biases, input x = (1, 0, 1), target y = 1.

In [ ]:
w_i, w_h, w_o = 0.5, 0.8, 1.0
x, y = [1.0, 0.0, 1.0], 1.0

h = [0.0]                                    # h_0 = 0
for x_t in x:
    h.append(np.tanh(w_i * x_t + w_h * h[-1]))
y_hat = 1 / (1 + np.exp(-w_o * h[3]))
loss = -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
print("h_1, h_2, h_3 =", np.round(h[1:], 4), " y_hat =", round(y_hat, 4), " loss =", round(loss, 4))

In [ ]:
# Backward. Sigmoid + binary cross-entropy: dL/dz = y_hat - y, where z = w_o * h_3
dz = y_hat - y
d = {k: 1 - h[k] ** 2 for k in (1, 2, 3)}          # tanh slope at each step
dh3_dh2, dh2_dh1 = d[3] * w_h, d[2] * w_h           # how h_t reacts to h_(t-1)
dL_dh3 = dz * w_o

g_o = dz * h[3]
# W_i: three paths. Each ends with the "immediate" derivative of h_k with respect to w_i: tanh slope * x_k
terms_i = [dL_dh3 * d[3] * x[2],                       # path through h_3 only
           dL_dh3 * dh3_dh2 * d[2] * x[1],             # path through h_3 and h_2
           dL_dh3 * dh3_dh2 * dh2_dh1 * d[1] * x[0]]   # path through h_3, h_2 and h_1
# W_h: same paths; the immediate derivative is tanh slope * h_(k-1)
terms_h = [dL_dh3 * d[3] * h[2],
           dL_dh3 * dh3_dh2 * d[2] * h[1],
           dL_dh3 * dh3_dh2 * dh2_dh1 * d[1] * h[0]]
print("dL/dz =", round(dz, 4), " tanh slopes:", {k: round(v, 4) for k, v in d.items()})
print("dh3/dh2 =", round(dh3_dh2, 4), " dh2/dh1 =", round(dh2_dh1, 4))
print("dL/dw_o =", round(g_o, 4))
print("dL/dw_i terms (t=3, 2, 1):", np.round(terms_i, 4), " sum =", round(sum(terms_i), 4))
print("dL/dw_h terms (t=3, 2, 1):", np.round(terms_h, 4), " sum =", round(sum(terms_h), 4))

In [ ]:
# TensorFlow computes the same gradients automatically
v_i, v_h, v_o = (tf.Variable(v, dtype=tf.float64) for v in (w_i, w_h, w_o))
with tf.GradientTape() as tape:
    s = tf.constant(0.0, tf.float64)
    for x_t in x:
        s = tf.tanh(v_i * x_t + v_h * s)
    p = tf.sigmoid(v_o * s)
    L = -tf.math.log(p)
g = tape.gradient(L, [v_i, v_h, v_o])
print("TensorFlow:", [round(float(t), 4) for t in g])
assert np.allclose([float(t) for t in g], [sum(terms_i), sum(terms_h), g_o])

rows = [dict(quantity=n, value=v) for n, v in [("h_1", h[1]), ("h_2", h[2]), ("h_3", h[3]), ("y_hat", y_hat),
        ("loss", loss), ("dL/dz", dz), ("dL/dw_o", g_o), ("dL/dw_i", sum(terms_i)), ("dL/dw_h", sum(terms_h))]]
rows += [dict(quantity=f"dL/dw_i term t={t}", value=v) for t, v in zip((3, 2, 1), terms_i)]
rows += [dict(quantity=f"dL/dw_h term t={t}", value=v) for t, v in zip((3, 2, 1), terms_h)]
pd.DataFrame(rows).round(6).to_csv(DATA / "worked_example.csv", index=False)

## 2. The 3-unit RNN on the toy reviews
Vocabulary cat, mat, rat; three reviews; SimpleRNN(3) without bias + Dense(1) without bias, as in the Note.

In [ ]:
vocab = ["cat", "mat", "rat"]
reviews = ["cat mat rat", "rat rat mat", "mat mat cat"]
Y = np.array([1, 1, 0], dtype="float32")
X = np.array([[np.eye(3)[vocab.index(w)] for w in r.split()] for r in reviews], dtype="float32")
print("X shape:", X.shape)

model = keras.Sequential([keras.Input(shape=(3, 3)),
                          keras.layers.SimpleRNN(3, use_bias=False),
                          keras.layers.Dense(1, activation="sigmoid", use_bias=False)])
model.summary()
W_i, W_h, W_o = [w.astype("float64") for w in model.get_weights()]


def bptt(x, y, W_i, W_h, W_o):
    """Forward pass, then BPTT for one review (rows are time steps). Returns loss and the 3 gradients."""
    T = len(x)
    hs = [np.zeros(3)]
    for t in range(T):
        hs.append(np.tanh(x[t] @ W_i + hs[-1] @ W_h))
    p = 1 / (1 + np.exp(-(hs[T] @ W_o)))
    L = -(y * np.log(p) + (1 - y) * np.log(1 - p)).item()
    dz = p - y                                       # (1,)
    gW_o = np.outer(hs[T], dz)
    gW_i, gW_h = np.zeros_like(W_i), np.zeros_like(W_h)
    dh = dz @ W_o.T                                  # dL/dh_T
    for t in range(T, 0, -1):                        # walk backwards in time
        da = dh * (1 - hs[t] ** 2)                   # through tanh at step t
        gW_i += np.outer(x[t - 1], da)               # this step's term for W_i
        gW_h += np.outer(hs[t - 1], da)              # this step's term for W_h
        dh = da @ W_h.T                              # pass the error back to h_(t-1)
    return L, gW_i, gW_h, gW_o


L0, gW_i, gW_h, gW_o = bptt(X[0], Y[0], W_i, W_h, W_o)
with tf.GradientTape() as tape:
    loss_tf = keras.losses.binary_crossentropy(Y[:1, None], model(X[:1]))
tf_grads = tape.gradient(loss_tf, model.trainable_variables)
for name, mine, theirs in zip(["W_i", "W_h", "W_o"], [gW_i, gW_h, gW_o], tf_grads):
    print(f"{name}: max difference NumPy vs TensorFlow = {np.abs(mine - theirs.numpy()).max():.1e}")
    assert np.allclose(mine, theirs.numpy(), atol=1e-5)

## 3. A real review: one gradient term per time step
Goodfellow §10.2.2 gives each time step its own copy W_h(t) of the shared matrix. The gradient of the shared
W_h is the sum of the gradients of the copies. We check that on the first IMDB test review (last 100 words).

In [ ]:
(_, _), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=10000)
review = keras.utils.pad_sequences(x_test[:1], maxlen=100)          # 100 time steps
print("review length used:", review.shape[1], " label:", y_test[0])

emb = keras.layers.Embedding(10000, 8)
rnn = keras.layers.SimpleRNN(16)
out = keras.layers.Dense(1, activation="sigmoid")
net = keras.Sequential([keras.Input(shape=(100,)), emb, rnn, out])
W_i_t, W_h_t, b_t = rnn.cell.kernel, rnn.cell.recurrent_kernel, rnn.cell.bias

with tf.GradientTape(persistent=True) as tape:
    e = emb(review)                                     # (1, 100, 8)
    copies = [tf.identity(W_h_t) for _ in range(100)]   # W_h(1), ..., W_h(100)
    for c in copies:
        tape.watch(c)
    s = tf.zeros((1, 16))
    for t in range(100):
        s = tf.tanh(e[:, t] @ W_i_t + s @ copies[t] + b_t)
    loss_manual = keras.losses.binary_crossentropy([[float(y_test[0])]], out(s))
    loss_keras = keras.losses.binary_crossentropy([[float(y_test[0])]], net(review))
per_step = tape.gradient(loss_manual, copies)
shared = tape.gradient(loss_keras, W_h_t)
diff = np.abs(np.sum([g.numpy() for g in per_step], axis=0) - shared.numpy()).max()
print(f"loss manual {float(loss_manual[0]):.6f}, Keras {float(loss_keras[0]):.6f}")
print(f"largest difference between the sum of 100 per-step terms and Keras' gradient: {diff:.1e}")
assert diff < 1e-6

## 4. Gradient descent with BPTT gradients
Plain gradient descent on the three toy reviews, using the NumPy BPTT above.

In [ ]:
lr, history = 0.5, []
for epoch in range(300):
    total, G = 0.0, [np.zeros_like(W_i), np.zeros_like(W_h), np.zeros_like(W_o)]
    for xr, yr in zip(X, Y):
        L, *gs = bptt(xr, yr, W_i, W_h, W_o)
        total += L / 3
        G = [a + b / 3 for a, b in zip(G, gs)]
    history.append(total)
    W_i, W_h, W_o = W_i - lr * G[0], W_h - lr * G[1], W_o - lr * G[2]
print(f"mean loss: epoch 1 {history[0]:.3f}, epoch 300 {history[-1]:.3f}")
preds = [1 / (1 + np.exp(-(np.tanh(np.tanh(np.tanh(xr[0] @ W_i) @ W_h + xr[1] @ W_i) @ W_h + xr[2] @ W_i) @ W_o)))
         for xr in X]
print("predictions after training:", np.round(np.ravel(preds), 3), " targets:", Y)

## Longer sequences: one term per time step, and the far ones shrink
The one-node example again ($w_i = 0.5$, $w_h = 0.8$, $w_o = 1$, target 1), now with every input equal to 1 and sequences of 3, 10 and 30 steps. For each time step we compute its term of $\partial L/\partial w_i$.

In [ ]:
import numpy as np, pandas as pd
rows = []
for T in (3, 10, 30):
    h = [0.0]
    for _ in range(T):
        h.append(np.tanh(0.5 * 1.0 + 0.8 * h[-1]))                 # forward through time
    y_hat = 1 / (1 + np.exp(-1.0 * h[-1]))
    delta = (y_hat - 1.0) * 1.0                                      # error reaching the last hidden state
    for t in range(T, 0, -1):                                        # backward through time
        a = delta * (1 - h[t] ** 2)
        rows.append(dict(T=T, step=t, steps_back=T - t, term=a * 1.0))   # this step's term: a * x_t
        delta = a * 0.8
terms = pd.DataFrame(rows)
terms.round(8).to_csv("data/path_terms_by_length.csv", index=False)
print(terms.groupby("T").term.agg(["count", "sum"]).round(4))
print(terms[terms["T"] == 30].iloc[[0, 1, 2, 5, 10, 29]].round(6).to_string(index=False))